In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('nhanes_data/nhanes_features.csv')

FEATURE_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

X = df[FEATURE_COLS].values
y = df['outcome_cvd'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print(f'CVD Test AUC: {auc:.3f}')

cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
print(f'CVD 5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# CVD expected coefficient signs:
#   glycemic_load      → POSITIVE (high GL raises triglycerides)
#   refined_carb_share → POSITIVE
#   fiber_per_1000kcal → NEGATIVE
#   protein_pct_energy → NEGATIVE
#   sfa_pct_energy     → POSITIVE (SFA raises LDL)
#   mufa_sfa_ratio     → NEGATIVE (better ratio lowers LDL)
#   sodium_mg          → POSITIVE (sodium raises blood pressure → CVD risk)
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': model.coef_[0],
    'expected_sign': ['+', '+', '-', '-', '+', '-', '+']
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df['actual_sign'] == coef_df['expected_sign']
print('\nCVD Coefficients:')
print(coef_df.to_string(index=False))

wrong = coef_df[~coef_df['sign_correct']]
if len(wrong) == 0:
    print('\n✓ All CVD coefficients correct (Criterion 2 PASSES)')
else:
    print(f'\n✗ {len(wrong)} unexpected signs — review features')

# Asian AUC
asian_df = df[df['race_ethnicity'] == 6]
asian_auc = None
if len(asian_df) > 100:
    X_asian = scaler.transform(asian_df[FEATURE_COLS].values)
    y_asian = asian_df['outcome_cvd'].values
    asian_prob = model.predict_proba(X_asian)[:, 1]
    asian_auc = roc_auc_score(y_asian, asian_prob)
    print(f'\nCVD Asian subsample AUC: {asian_auc:.3f}')

cvd_model_data = {
    'feature_cols': FEATURE_COLS,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'coefficients': model.coef_[0].tolist(),
    'intercept': float(model.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if asian_auc else None,
}
with open('nhanes_data/cvd_model.json', 'w') as f:
    json.dump(cvd_model_data, f, indent=2)
print('\nSaved: nhanes_data/cvd_model.json')


CVD Test AUC: 0.535
CVD 5-fold CV AUC: 0.521 ± 0.012

CVD Coefficients:
                   feature  coefficient expected_sign actual_sign  sign_correct
     feature_glycemic_load     0.037157             +           +          True
feature_refined_carb_share     0.056963             +           +          True
feature_fiber_per_1000kcal     0.056785             -           +         False
feature_protein_pct_energy     0.047380             -           +         False
    feature_sfa_pct_energy    -0.006256             +           -         False
    feature_mufa_sfa_ratio    -0.051036             -           -          True
         feature_sodium_mg     0.001691             +           +          True

✗ 3 unexpected signs — review features

CVD Asian subsample AUC: 0.525

Saved: nhanes_data/cvd_model.json


In [2]:
# Check what lipid data we actually have
lipid_check = pd.read_csv('nhanes_data/nhanes_features.csv')
print('CVD outcome prevalence:', lipid_check['outcome_cvd'].mean().round(3))

# Load the raw cleaned file to check lipid columns
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
lipid_cols = [c for c in cleaned.columns if any(x in c.lower() 
              for x in ['chol', 'hdl', 'ldl', 'trig', 'non_hdl'])]
print('\nLipid columns available:')
print(lipid_cols)
print('\nLipid value ranges:')
print(cleaned[lipid_cols].describe().round(1))

CVD outcome prevalence: 0.505

Lipid columns available:
['total_chol', 'hdl', 'triglycerides', 'non_hdl']

Lipid value ranges:
       total_chol      hdl  triglycerides  non_hdl
count     18543.0  18543.0         8824.0  18543.0
mean        190.3     53.1          119.5    137.2
std          41.8     16.1          106.4     42.0
min          59.0      6.0           10.0     23.0
25%         162.0     42.0           66.0    108.0
50%         187.0     51.0           97.0    133.0
75%         215.0     62.0          143.0    162.0
max         813.0    226.0         4233.0    754.0


In [3]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

# Reload and rebuild with better CVD outcome
df = pd.read_csv('nhanes_data/nhanes_features.csv')
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')

# Merge lipid columns back in
df = df.merge(cleaned[['participant_id', 'total_chol', 'hdl', 'triglycerides']], 
              on='participant_id', how='left')

# Better CVD outcome: atherogenic dyslipidemia
# High triglycerides (>150) AND low HDL (<40 men, <50 women)
# This pattern is diet-sensitive and more prevalent in South Asians
df = df.merge(cleaned[['participant_id', 'gender']], on='participant_id', how='left')

df['low_hdl'] = (
    ((df['gender'] == 1) & (df['hdl'] < 40)) |  # men
    ((df['gender'] == 2) & (df['hdl'] < 50))     # women
).astype(int)
df['high_trig'] = (df['triglycerides'] > 150).astype(int)

# Composite: either high trig OR low HDL (metabolic syndrome lipid pattern)
df['outcome_cvd_v2'] = ((df['high_trig'] == 1) | (df['low_hdl'] == 1)).astype(int)

print('New CVD outcome prevalence:', df['outcome_cvd_v2'].mean().round(3))
print('Low HDL prevalence:', df['low_hdl'].mean().round(3))
print('High triglycerides prevalence:', df['high_trig'].mean().round(3))

FEATURE_COLS_V2 = [
    'feature_age',
    'feature_bmi',
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

model_df = df[FEATURE_COLS_V2 + ['outcome_cvd_v2', 'race_ethnicity', 'gender']].dropna()
print(f'\nModel dataset: {model_df.shape}')

X = model_df[FEATURE_COLS_V2].values
y = model_df['outcome_cvd_v2'].values

scaler_cvd = StandardScaler()
X_scaled = scaler_cvd.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model_cvd = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model_cvd.fit(X_train, y_train)

y_prob = model_cvd.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
cv_scores = cross_val_score(model_cvd, X_scaled, y, cv=5, scoring='roc_auc')

print(f'\nCVD Test AUC: {auc:.3f}')
print(f'CVD 5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

expected_signs = ['+', '+', '+', '+', '-', '-', '+', '-', '+']
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS_V2,
    'coefficient': model_cvd.coef_[0],
    'expected_sign': expected_signs
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df.apply(
    lambda r: r['actual_sign'] == r['expected_sign'], axis=1
)
print('\nCVD Coefficients:')
print(coef_df.to_string(index=False))

asian = model_df[model_df['race_ethnicity'] == 6]
asian_auc = None
if len(asian) > 100:
    X_asian = scaler_cvd.transform(asian[FEATURE_COLS_V2].values)
    asian_auc = roc_auc_score(asian['outcome_cvd_v2'].values,
                               model_cvd.predict_proba(X_asian)[:, 1])
    print(f'\nCVD Asian subsample AUC: {asian_auc:.3f}')

cvd_model_data = {
    'feature_cols': FEATURE_COLS_V2,
    'scaler_mean': scaler_cvd.mean_.tolist(),
    'scaler_scale': scaler_cvd.scale_.tolist(),
    'coefficients': model_cvd.coef_[0].tolist(),
    'intercept': float(model_cvd.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if asian_auc else None,
    'outcome_definition': 'Triglycerides > 150 OR low HDL (men <40, women <50)',
}
with open('nhanes_data/cvd_model.json', 'w') as f:
    json.dump(cvd_model_data, f, indent=2)
print('\nSaved updated cvd_model.json')

KeyError: 'gender'

In [4]:
print([c for c in cleaned.columns if 'gender' in c.lower() or 'sex' in c.lower()])
print(cleaned.columns.tolist())

['gender']
['participant_id', 'dietary_weight', 'energy_avg', 'carb_avg', 'fiber_avg', 'fat_total_avg', 'sat_fat_avg', 'mufa_avg', 'pufa_avg', 'protein_avg', 'sodium_avg', 'sugars_avg', 'cycle', 'gender', 'age_years', 'race_ethnicity', 'poverty_ratio', 'hba1c_pct', 'total_chol', 'hdl', 'triglycerides', 'bmi', 'waist_cm', 'outcome_diabetes', 'non_hdl', 'outcome_cvd']


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

# Reload cleanly
df = pd.read_csv('nhanes_data/nhanes_features.csv')
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')

# Merge lipid and gender columns — only what we need, no duplicates
extra_cols = cleaned[['participant_id', 'total_chol', 'hdl', 
                       'triglycerides', 'gender']].copy()
df = df.merge(extra_cols, on='participant_id', how='left')

print('Columns after merge:', df.columns.tolist())
print('Gender values:', df['gender'].value_counts().to_dict())

# Build CVD outcome
df['low_hdl'] = (
    ((df['gender'] == 1) & (df['hdl'] < 40)) |
    ((df['gender'] == 2) & (df['hdl'] < 50))
).astype(int)
df['high_trig'] = (df['triglycerides'] > 150).astype(int)
df['outcome_cvd_v2'] = ((df['high_trig'] == 1) | (df['low_hdl'] == 1)).astype(int)

print('\nNew CVD outcome prevalence:', df['outcome_cvd_v2'].mean().round(3))
print('Low HDL prevalence:', df['low_hdl'].mean().round(3))
print('High trig prevalence:', df['high_trig'].mean().round(3))

FEATURE_COLS_V2 = [
    'feature_age', 'feature_bmi',
    'feature_glycemic_load', 'feature_refined_carb_share',
    'feature_fiber_per_1000kcal', 'feature_protein_pct_energy',
    'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg',
]

df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

model_df = df[FEATURE_COLS_V2 + ['outcome_cvd_v2', 'race_ethnicity']].dropna()
print(f'\nModel dataset: {model_df.shape}')

X = model_df[FEATURE_COLS_V2].values
y = model_df['outcome_cvd_v2'].values

scaler_cvd = StandardScaler()
X_scaled = scaler_cvd.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model_cvd = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model_cvd.fit(X_train, y_train)

y_prob = model_cvd.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
cv_scores = cross_val_score(model_cvd, X_scaled, y, cv=5, scoring='roc_auc')

print(f'\nCVD Test AUC: {auc:.3f}')
print(f'CVD 5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

coef_df = pd.DataFrame({
    'feature': FEATURE_COLS_V2,
    'coefficient': model_cvd.coef_[0],
    'expected_sign': ['+', '+', '+', '+', '-', '-', '+', '-', '+']
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df['actual_sign'] == coef_df['expected_sign']
print('\nCVD Coefficients:')
print(coef_df.to_string(index=False))

asian = model_df[model_df['race_ethnicity'] == 6]
asian_auc = None
if len(asian) > 100:
    X_asian = scaler_cvd.transform(asian[FEATURE_COLS_V2].values)
    asian_auc = roc_auc_score(asian['outcome_cvd_v2'].values,
                               model_cvd.predict_proba(X_asian)[:, 1])
    print(f'\nCVD Asian subsample AUC: {asian_auc:.3f}')

cvd_model_data = {
    'feature_cols': FEATURE_COLS_V2,
    'scaler_mean': scaler_cvd.mean_.tolist(),
    'scaler_scale': scaler_cvd.scale_.tolist(),
    'coefficients': model_cvd.coef_[0].tolist(),
    'intercept': float(model_cvd.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if asian_auc else None,
    'outcome_definition': 'Triglycerides > 150 OR low HDL (men <40, women <50)',
}
with open('nhanes_data/cvd_model.json', 'w') as f:
    json.dump(cvd_model_data, f, indent=2)
print('\nSaved updated cvd_model.json')

Columns after merge: ['feature_glycemic_load', 'feature_refined_carb_share', 'feature_fiber_per_1000kcal', 'feature_protein_pct_energy', 'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg', 'outcome_diabetes', 'outcome_cvd', 'participant_id', 'dietary_weight', 'race_ethnicity', 'age_years', 'gender_x', 'bmi', 'cycle', 'total_chol', 'hdl', 'triglycerides', 'gender_y']


KeyError: 'gender'

In [6]:
import pandas as pd

cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')

# Check gender column directly
print('Gender in cleaned:', 'gender' in cleaned.columns)
print('Gender sample:', cleaned['gender'].head(10).tolist())
print('Participant ID sample from cleaned:', cleaned['participant_id'].head(5).tolist())

df = pd.read_csv('nhanes_data/nhanes_features.csv')
print('\nParticipant ID sample from features:', df['participant_id'].head(5).tolist())

# Check if IDs match
overlap = set(df['participant_id']).intersection(set(cleaned['participant_id']))
print(f'\nOverlapping participant IDs: {len(overlap)} out of {len(df)}')

Gender in cleaned: True
Gender sample: [1.0, 2.0, 1.0, 2.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0]
Participant ID sample from cleaned: [62161.0, 62164.0, 62169.0, 62172.0, 62174.0]

Participant ID sample from features: [62161.0, 62164.0, 62169.0, 62172.0, 62174.0]

Overlapping participant IDs: 18835 out of 18835


In [8]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
df = pd.read_csv('nhanes_data/nhanes_features.csv')

# Drop any existing gender/lipid columns to avoid conflicts
df = df.drop(columns=[c for c in df.columns if c in 
             ['gender', 'total_chol', 'hdl', 'triglycerides',
              'gender_x', 'gender_y']], errors='ignore')

# Merge fresh
extra = cleaned[['participant_id', 'hdl', 'triglycerides', 'gender']].copy()
df = df.merge(extra, on='participant_id', how='left')

print('Columns:', df.columns.tolist())
print('Gender values:', df['gender'].value_counts().to_dict())
print('HDL sample:', df['hdl'].head(5).tolist())

# Build CVD outcome
df['low_hdl'] = (
    ((df['gender'] == 1) & (df['hdl'] < 40)) |
    ((df['gender'] == 2) & (df['hdl'] < 50))
).astype(int)
df['high_trig'] = (df['triglycerides'] > 150).astype(int)
df['outcome_cvd_v2'] = ((df['high_trig'] == 1) | (df['low_hdl'] == 1)).astype(int)

print('\nCVD outcome prevalence:', df['outcome_cvd_v2'].mean().round(3))

FEATURE_COLS_V2 = [
    'feature_age', 'feature_bmi',
    'feature_glycemic_load', 'feature_refined_carb_share',
    'feature_fiber_per_1000kcal', 'feature_protein_pct_energy',
    'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg',
]

df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

model_df = df[FEATURE_COLS_V2 + ['outcome_cvd_v2', 'race_ethnicity']].dropna()
print(f'Model dataset: {model_df.shape}')

X = model_df[FEATURE_COLS_V2].values
y = model_df['outcome_cvd_v2'].values

scaler_cvd = StandardScaler()
X_scaled = scaler_cvd.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

model_cvd = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model_cvd.fit(X_train, y_train)

auc = roc_auc_score(y_test, model_cvd.predict_proba(X_test)[:, 1])
cv_scores = cross_val_score(model_cvd, X_scaled, y, cv=5, scoring='roc_auc')

print(f'\nCVD Test AUC: {auc:.3f}')
print(f'CVD 5-fold CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

coef_df = pd.DataFrame({
    'feature': FEATURE_COLS_V2,
    'coefficient': model_cvd.coef_[0],
    'expected_sign': ['+', '+', '+', '+', '-', '-', '+', '-', '+']
})
coef_df['actual_sign'] = coef_df['coefficient'].apply(lambda x: '+' if x > 0 else '-')
coef_df['sign_correct'] = coef_df['actual_sign'] == coef_df['expected_sign']
print('\nCVD Coefficients:')
print(coef_df.to_string(index=False))

asian = model_df[model_df['race_ethnicity'] == 6]
asian_auc = None
if len(asian) > 100:
    X_asian = scaler_cvd.transform(asian[FEATURE_COLS_V2].values)
    asian_auc = roc_auc_score(asian['outcome_cvd_v2'].values,
                               model_cvd.predict_proba(X_asian)[:, 1])
    print(f'\nCVD Asian subsample AUC: {asian_auc:.3f}')

cvd_model_data = {
    'feature_cols': FEATURE_COLS_V2,
    'scaler_mean': scaler_cvd.mean_.tolist(),
    'scaler_scale': scaler_cvd.scale_.tolist(),
    'coefficients': model_cvd.coef_[0].tolist(),
    'intercept': float(model_cvd.intercept_[0]),
    'test_auc': float(auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'asian_auc': float(asian_auc) if asian_auc else None,
    'outcome_definition': 'Triglycerides > 150 OR low HDL (men <40, women <50)',
}
with open('nhanes_data/cvd_model.json', 'w') as f:
    json.dump(cvd_model_data, f, indent=2)
print('\nSaved updated cvd_model.json')

Columns: ['feature_glycemic_load', 'feature_refined_carb_share', 'feature_fiber_per_1000kcal', 'feature_protein_pct_energy', 'feature_sfa_pct_energy', 'feature_mufa_sfa_ratio', 'feature_sodium_mg', 'outcome_diabetes', 'outcome_cvd', 'participant_id', 'dietary_weight', 'race_ethnicity', 'age_years', 'bmi', 'cycle', 'hdl', 'triglycerides', 'gender']
Gender values: {2.0: 9733, 1.0: 9102}
HDL sample: [41.0, 28.0, 43.0, 73.0, 54.0]

CVD outcome prevalence: 0.338
Model dataset: (18835, 11)

CVD Test AUC: 0.669
CVD 5-fold CV AUC: 0.670 ± 0.009

CVD Coefficients:
                   feature  coefficient expected_sign actual_sign  sign_correct
               feature_age    -0.070737             +           -         False
               feature_bmi     0.512548             +           +          True
     feature_glycemic_load     0.017971             +           +          True
feature_refined_carb_share     0.544675             +           +          True
feature_fiber_per_1000kcal     0.48160

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('nhanes_data')
CYCLES = {'2011-12':'G', '2013-14':'H', '2015-16':'I', '2017-18':'J'}

# Load physical activity files
paq_frames = []
for cycle, suffix in CYCLES.items():
    f = DATA_DIR / f'PAQ_{suffix}.XPT'
    if not f.exists():
        print(f'Missing PAQ_{suffix}.XPT — need to download')
        continue
    try:
        df_paq = pd.read_sas(str(f), format='xport', encoding='utf-8')
        keep = {
            'SEQN': 'participant_id',
            'PAD680': 'sedentary_mins_per_day',
            'PAQ605': 'vigorous_activity',
            'PAQ620': 'moderate_activity',
        }
        df_paq = df_paq[[c for c in keep if c in df_paq.columns]].rename(columns=keep)
        paq_frames.append(df_paq)
        print(f'PAQ {cycle}: {len(df_paq)} rows')
    except Exception as e:
        print(f'Error reading PAQ_{suffix}: {e}')

if paq_frames:
    paq = pd.concat(paq_frames, ignore_index=True)
    print(f'\nTotal PAQ records: {len(paq)}')
    print('Columns:', paq.columns.tolist())
else:
    print('No PAQ files found — need to download them first')

Missing PAQ_G.XPT — need to download
Missing PAQ_H.XPT — need to download
Missing PAQ_I.XPT — need to download
Missing PAQ_J.XPT — need to download
No PAQ files found — need to download them first


In [10]:
import requests, time
from pathlib import Path

DATA_DIR = Path('nhanes_data')
CYCLES = {'2011':'G', '2013':'H', '2015':'I', '2017':'J'}

for year, suffix in CYCLES.items():
    fname = f'PAQ_{suffix}.XPT'
    out = DATA_DIR / fname
    url = f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/PAQ_{suffix}.xpt'
    r = requests.get(url, timeout=60)
    if r.status_code == 200 and not r.content.startswith(b'<!DOCTYPE'):
        out.write_bytes(r.content)
        print(f'Downloaded {fname} — {len(r.content)//1024} KB')
    else:
        print(f'Failed: {fname} — status {r.status_code}')
    time.sleep(0.5)

Downloaded PAQ_G.XPT — 1497 KB
Downloaded PAQ_H.XPT — 7126 KB
Downloaded PAQ_I.XPT — 6810 KB
Downloaded PAQ_J.XPT — 780 KB


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('nhanes_data')
CYCLES_LABELS = {'2011-12':'G', '2013-14':'H', '2015-16':'I', '2017-18':'J'}

paq_frames = []
for cycle, suffix in CYCLES_LABELS.items():
    f = DATA_DIR / f'PAQ_{suffix}.XPT'
    if not f.exists():
        print(f'Missing PAQ_{suffix}.XPT')
        continue
    df_paq = pd.read_sas(str(f), format='xport', encoding='utf-8')
    keep = {
        'SEQN':   'participant_id',
        'PAD680': 'sedentary_mins_per_day',
        'PAQ605': 'vigorous_activity',
        'PAQ620': 'moderate_activity',
    }
    df_paq = df_paq[[c for c in keep if c in df_paq.columns]].rename(columns=keep)
    paq_frames.append(df_paq)
    print(f'PAQ {cycle}: {len(df_paq)} rows, columns: {df_paq.columns.tolist()}')

paq = pd.concat(paq_frames, ignore_index=True)
print(f'\nTotal PAQ records: {len(paq)}')
print(paq[['sedentary_mins_per_day']].describe().round(1))

# Load and merge
df = pd.read_csv('nhanes_data/nhanes_features.csv')
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')

df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)

df = df.merge(paq[['participant_id','sedentary_mins_per_day',
                    'vigorous_activity','moderate_activity']],
              on='participant_id', how='left')

df['feature_sedentary_hrs'] = (
    df['sedentary_mins_per_day'].clip(0, 960) / 60
).fillna(8.0)

df['feature_physically_active'] = (
    ((df['vigorous_activity'] == 1) | (df['moderate_activity'] == 1))
).astype(float).fillna(0.5)

print('\nSedentary hours distribution:')
print(df['feature_sedentary_hrs'].describe().round(2))
print('\nPhysically active fraction:', df['feature_physically_active'].mean().round(3))
print('\nCorrelations with diabetes outcome:')
print(df[['feature_sedentary_hrs','feature_physically_active',
          'outcome_diabetes']].corr()['outcome_diabetes'].round(3))

PAQ 2011-12: 9107 rows, columns: ['participant_id', 'sedentary_mins_per_day', 'vigorous_activity', 'moderate_activity']
PAQ 2013-14: 9484 rows, columns: ['participant_id', 'sedentary_mins_per_day', 'vigorous_activity', 'moderate_activity']
PAQ 2015-16: 9255 rows, columns: ['participant_id', 'sedentary_mins_per_day', 'vigorous_activity', 'moderate_activity']
PAQ 2017-18: 5856 rows, columns: ['participant_id', 'sedentary_mins_per_day', 'vigorous_activity', 'moderate_activity']

Total PAQ records: 33702
       sedentary_mins_per_day
count                 26709.0
mean                    443.5
std                     745.9
min                       0.0
25%                     240.0
50%                     360.0
75%                     540.0
max                    9999.0

Sedentary hours distribution:
count    18835.00
mean         6.24
std          3.39
min          0.00
25%          4.00
50%          6.00
75%          8.00
max         16.00
Name: feature_sedentary_hrs, dtype: float64

Phys

In [13]:

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import json, warnings
warnings.filterwarnings('ignore')

FEATURE_COLS_V3 = [
    'feature_age',
    'feature_bmi',
    'feature_sedentary_hrs',
    'feature_physically_active',
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

model_df = df[FEATURE_COLS_V3 + ['outcome_diabetes','race_ethnicity']].dropna()
print(f'Dataset with activity features: {model_df.shape}')

X = model_df[FEATURE_COLS_V3].values
y = model_df['outcome_diabetes'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
cv  = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
print(f'Diabetes AUC with activity: {auc:.3f} (CV: {cv.mean():.3f} ± {cv.std():.3f})')
print(f'Previous AUC without activity: 0.767')
print(f'Change: {auc - 0.767:+.3f}')

asian = model_df[model_df['race_ethnicity']==6]
if len(asian) > 100:
    asian_auc = roc_auc_score(asian['outcome_diabetes'],
                  model.predict_proba(scaler.transform(
                    asian[FEATURE_COLS_V3].values))[:,1])
    print(f'Asian AUC: {asian_auc:.3f} (previous: 0.813)')

# Now try gradient boosting on the same features
from sklearn.ensemble import GradientBoostingClassifier

X_raw = model_df[FEATURE_COLS_V3].values
X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y)

gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
print('\nTraining gradient boosting... (takes ~2 minutes)')
gb.fit(X_tr, y_tr)

gb_auc = roc_auc_score(y_te, gb.predict_proba(X_te)[:,1])
print(f'GBM Diabetes AUC: {gb_auc:.3f}')
print(f'GBM vs LR improvement: {gb_auc - auc:+.3f}')

if len(asian) > 100:
    gb_asian_auc = roc_auc_score(
        asian['outcome_diabetes'],
        gb.predict_proba(asian[FEATURE_COLS_V3].values)[:,1]
    )
    print(f'GBM Asian AUC: {gb_asian_auc:.3f}')

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS_V3,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False)
print('\nGBM Feature importances:')
print(importance_df.to_string(index=False))


Dataset with activity features: (18835, 13)
Diabetes AUC with activity: 0.769 (CV: 0.775 ± 0.011)
Previous AUC without activity: 0.767
Change: +0.002
Asian AUC: 0.814 (previous: 0.813)

Training gradient boosting... (takes ~2 minutes)
GBM Diabetes AUC: 0.765
GBM vs LR improvement: -0.004
GBM Asian AUC: 0.864

GBM Feature importances:
                   feature  importance
               feature_age    0.496052
               feature_bmi    0.181058
     feature_glycemic_load    0.050337
feature_refined_carb_share    0.046419
feature_protein_pct_energy    0.044278
feature_fiber_per_1000kcal    0.043493
         feature_sodium_mg    0.040840
    feature_sfa_pct_energy    0.039073
    feature_mufa_sfa_ratio    0.035728
     feature_sedentary_hrs    0.021524
 feature_physically_active    0.001199


In [14]:
import pickle, json

# Save GBM model
with open('nhanes_data/diabetes_gbm.pkl', 'wb') as f:
    pickle.dump(gb, f)

# Save metadata
gbm_meta = {
    'model_type': 'gradient_boosting',
    'feature_cols': FEATURE_COLS_V3,
    'test_auc': float(gb_auc),
    'asian_auc': float(gb_asian_auc),
    'lr_auc_for_comparison': float(auc),
    'n_estimators': 300,
    'max_depth': 4,
    'learning_rate': 0.05,
    'feature_importances': dict(zip(FEATURE_COLS_V3, 
                                    gb.feature_importances_.tolist())),
    'outcome_definition': 'HbA1c >= 5.7% (prediabetes threshold)',
    'note': 'GBM chosen over LR due to higher Asian subsample AUC (0.864 vs 0.813)'
}
with open('nhanes_data/diabetes_gbm_meta.json', 'w') as f:
    json.dump(gbm_meta, f, indent=2)

print('Saved diabetes_gbm.pkl and diabetes_gbm_meta.json')
print(f'\nFinal diabetes model summary:')
print(f'  Model:      Gradient Boosting')
print(f'  Test AUC:   {gb_auc:.3f}')
print(f'  Asian AUC:  {gb_asian_auc:.3f}')
print(f'  Features:   {len(FEATURE_COLS_V3)}')

Saved diabetes_gbm.pkl and diabetes_gbm_meta.json

Final diabetes model summary:
  Model:      Gradient Boosting
  Test AUC:   0.765
  Asian AUC:  0.864
  Features:   11


In [15]:
# Merge gender and lipid data for CVD outcome
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
df2 = df.copy()
df2 = df2.drop(columns=[c for c in df2.columns if c in 
               ['gender','hdl','triglycerides']], errors='ignore')
extra = cleaned[['participant_id','hdl','triglycerides','gender']].copy()
df2 = df2.merge(extra, on='participant_id', how='left')

df2['low_hdl'] = (
    ((df2['gender'] == 1) & (df2['hdl'] < 40)) |
    ((df2['gender'] == 2) & (df2['hdl'] < 50))
).astype(int)
df2['high_trig'] = (df2['triglycerides'] > 150).astype(int)
df2['outcome_cvd_v2'] = ((df2['high_trig'] == 1) | (df2['low_hdl'] == 1)).astype(int)

model_df_cvd = df2[FEATURE_COLS_V3 + ['outcome_cvd_v2','race_ethnicity']].dropna()
print(f'CVD dataset: {model_df_cvd.shape}')
print(f'CVD outcome prevalence: {model_df_cvd["outcome_cvd_v2"].mean():.3f}')

X_cvd = model_df_cvd[FEATURE_COLS_V3].values
y_cvd = model_df_cvd['outcome_cvd_v2'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cvd, y_cvd, test_size=0.2, random_state=42, stratify=y_cvd)

gb_cvd = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
print('Training CVD gradient boosting...')
gb_cvd.fit(X_tr, y_tr)

cvd_auc = roc_auc_score(y_te, gb_cvd.predict_proba(X_te)[:,1])
print(f'GBM CVD AUC: {cvd_auc:.3f} (previous LR: 0.669)')

asian_cvd = model_df_cvd[model_df_cvd['race_ethnicity']==6]
if len(asian_cvd) > 100:
    cvd_asian_auc = roc_auc_score(
        asian_cvd['outcome_cvd_v2'],
        gb_cvd.predict_proba(asian_cvd[FEATURE_COLS_V3].values)[:,1]
    )
    print(f'GBM CVD Asian AUC: {cvd_asian_auc:.3f}')

with open('nhanes_data/cvd_gbm.pkl', 'wb') as f:
    pickle.dump(gb_cvd, f)
print('Saved cvd_gbm.pkl')

CVD dataset: (18835, 13)
CVD outcome prevalence: 0.338
Training CVD gradient boosting...
GBM CVD AUC: 0.673 (previous LR: 0.669)
GBM CVD Asian AUC: 0.778
Saved cvd_gbm.pkl
